# Build Auxiliary Model - Single `.mo` File

This notebook builds the auxiliary model from a single dynamic `.mo` file.

It takes the input model from `models/`, replaces each dynamic component with its static counterpart, adds the INIT model where there is one, removes the dynamic events, and saves the auxiliary model to `outputs/`.

In [ ]:
import logging
import os
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

from OMPython import OMCSessionLocal

from scripts.helpers import *

# --- Configuration ---

# 1. Directory containing the single-file model
MODEL_DIR = os.path.abspath("models")

# 2. Select the model to build
MODEL = "Demo_BESS"

# 3. Path to the selected model file
MODEL_FILE_PATH = os.path.join(MODEL_DIR, MODEL + ".mo")

# 4. Modelica Standard Library
OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")

# 5. Dynawo Modelica library, for now the one the Julia family downloads
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "..", "julia_openmodelica", "dynawo_library", "Dynawo", "package.mo")
)

# 6. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = {
    # "generatorSynchronous": "GeneratorSynchronousInt_INIT",
}

# 7. Slack component (leave empty to disable slack-specific handling).
SLACK_COMPONENT = ""

## Load and Validate the Model

The original dynamic model is loaded and checked, and the user configuration is validated against it, before any changes are made.

In [ ]:
# 1. Start OMC and load libraries
omc = OMCSessionLocal()
omc_call(omc, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(omc, "loadModel(Complex)")
omc_call(omc, "loadModel(ModelicaServices)")
omc_call(omc, f'loadFile("{DYNAWO_PKG_PATH}")')

# 2. Load the selected model and validate the configuration against it
omc_call(omc, f'loadFile("{MODEL_FILE_PATH}")')
check_user_configuration_single(
    omc,
    model=MODEL,
    slack_component=SLACK_COMPONENT,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)

## Generate the Auxiliary Model

The auxiliary model is generated from a copy of the original and saved to `outputs/`.

In [ ]:
OUTPUT_DIR = os.path.abspath("outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUX_MODEL = MODEL + "_auxiliary"
AUX_FILE = os.path.join(OUTPUT_DIR, AUX_MODEL + ".mo")

In [ ]:
# Create/refresh the auxiliary model in OpenModelica
send_expression(omc, f"deleteClass({AUX_MODEL})")
omc_call(omc, f'copyClass({MODEL}, "{AUX_MODEL}")')

# Build component dictionary from the source model
components = get_all_components(omc, MODEL)

# Apply dictionary-driven replacements
apply_replacements(omc, MODEL, AUX_MODEL, components, SLACK_COMPONENT)

# Delete connections to cleanup targets
delete_connections(omc, AUX_MODEL, components)

# Delete cleanup-target components
delete_components(omc, AUX_MODEL, components)

# Add INIT models for the source model
add_init_models(omc, MODEL, AUX_MODEL, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Add load-flow modifiers
apply_LF_modifiers(omc, MODEL, AUX_MODEL, components)

# Add initial equations for the source model
add_init_equations(omc, MODEL, AUX_MODEL, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Clean up the auxiliary model equations in place
clean_aux_equations(omc, AUX_MODEL, components, SLACK_COMPONENT)

# Validate the build
chk = send_expression(omc, f"checkModel({AUX_MODEL})", parsed=False)
print(chk)

# Save the auxiliary model
omc_call(omc, f'saveModel("{AUX_FILE}", {AUX_MODEL})')